## Sensitivity Analysis notebook

In [ ]:
import gc, sys,  pickle
from pathlib import Path
import numpy as np
import pandas as pd
import duckdb
from sklearn.metrics import cohen_kappa_score, confusion_matrix, roc_auc_score


In [ ]:
sys.path.append(str(Path.cwd()))

from kdigo_pipeline import (
    generate_urine_labels_elapsed_time, kdigo_scr_15x, kdigo_scr_03, kdigo_scr_03_early,
    combine_scr_labels, get_time_window,
    compute_baseline_scr, get_urine_evaluability_labels, compute_measurement_gaps,
    compute_weight_from_durations, compute_weight_table,
    compute_rolling_creatinine_baseline, kdigo_scr_15x_rolling, kdigo_scr_03_rolling
)

In [ ]:
PROJECT_DIR = Path.cwd()
DATA = PROJECT_DIR / "Mimic"

HOSP = DATA / "hosp"
ICU = DATA / "icu"
OUTPUT = PROJECT_DIR / "processed_outputs"
VARIANT_OUTPUT = OUTPUT / "Sensitivity_Analysis" / "variants"
VARIANT_OUTPUT.mkdir(parents=True, exist_ok=True)

#### Load shared raw data 

In [ ]:
cohort = pd.read_parquet(OUTPUT / "cohort.parquet")
cohort_stay_ids = set(cohort["stay_id"])
cohort["los_hours"] = (cohort["outtime"] - cohort["intime"]).dt.total_seconds() / 3600
print(cohort.shape, "| unique stays:", len(cohort_stay_ids))


In [ ]:
labevents_raw = pd.read_csv(DATA / "hosp/labevents.csv", usecols=["subject_id", "hadm_id", "itemid","charttime","valuenum"], parse_dates=["charttime"])


In [ ]:
labevents_with_hadm = labevents_raw.merge(
    cohort[["subject_id", "hadm_id", "stay_id", "intime"]],
    on=["subject_id", "hadm_id"], how="inner"
)


In [ ]:
labevents_no_hadm = labevents_raw[labevents_raw["hadm_id"].isna()].merge(
    cohort[["subject_id", "hadm_id", "stay_id", "intime"]].rename(columns={"hadm_id": "cohort_hadm_id"}),
    on="subject_id", how="inner"
)
labevents_no_hadm["hours_before_intime"] = (
    labevents_no_hadm["intime"] - labevents_no_hadm["charttime"]
).dt.total_seconds() / 3600
labevents_no_hadm = labevents_no_hadm[
    (labevents_no_hadm["hours_before_intime"] >= 0) &
    (labevents_no_hadm["hours_before_intime"] <= 168)
].drop(columns=["cohort_hadm_id", "hours_before_intime"])

In [ ]:
#de-duplicate multi-stay matches (keep closest stay in time, if subject has >1 stay)
dup_matches = labevents_no_hadm.groupby(["subject_id","charttime","itemid"]).size()
print("Null-hadm_id lab rows matching multiple stays:", (dup_matches > 1).sum())
labevents_no_hadm["time_to_intime"] = (labevents_no_hadm["intime"] - labevents_no_hadm["charttime"])
labevents_no_hadm = labevents_no_hadm.sort_values("time_to_intime").drop_duplicates(
    subset=["subject_id","charttime","itemid"], keep="first"
).drop(columns=["time_to_intime"])

labevents = pd.concat([labevents_with_hadm, labevents_no_hadm], ignore_index=True)
print("Labevents with hadm_id match:", len(labevents_with_hadm))
print("Labevents with null hadm_id, recovered:", len(labevents_no_hadm))
print("Cohort-restricted labevents (combined):", len(labevents))

In [ ]:
URINE_OUTPUT_ITEMIDS = [226557, 226558, 226559, 226560, 226561, 226563, 226564, 226565, 226567, 226584, 227488, 227489]
output_chunks =[]
for chunk in pd.read_csv(DATA / "icu/outputevents.csv", usecols=["subject_id","hadm_id", "stay_id", "charttime","itemid","value"], chunksize=500_000):
    chunk =chunk[chunk["itemid"].isin(URINE_OUTPUT_ITEMIDS)]
    if chunk.empty:
        continue
    chunk =chunk[chunk["stay_id"].isin(cohort_stay_ids)]
    if not chunk.empty:
        output_chunks.append(chunk)

print("Chunks collected:", len(output_chunks))
outputevents =pd.concat(output_chunks, ignore_index=True)
print("After concat:", outputevents.shape)

outputevents["charttime"] =pd.to_datetime(outputevents["charttime"], errors="coerce")
print("After datetime conversion:", outputevents.shape)

outputevents =outputevents.merge(cohort[["stay_id", "intime"]], on="stay_id", how="inner")
print("After intime merge:", outputevents.shape)

outputevents =compute_measurement_gaps(outputevents)
print("After compute_measurement_gaps:", outputevents.shape)


In [ ]:
WEIGHT_ITEMIDS =[ 226512,224639,226531] 
    

weight_chunks =[]

for chunk in pd.read_csv(DATA / "icu/chartevents.csv",usecols=["stay_id","charttime", "itemid","valuenum"],chunksize=500_000):

    chunk=chunk[chunk["itemid"].isin(WEIGHT_ITEMIDS)]
    if chunk.empty:
        continue
    chunk=chunk[chunk["stay_id"].isin(cohort_stay_ids)]
    if not chunk.empty:
        weight_chunks.append(chunk)
chartevents=pd.concat(weight_chunks,ignore_index=True)
del weight_chunks
gc.collect()

chartevents["charttime"] =pd.to_datetime(chartevents["charttime"],errors="coerce")
chartevents =chartevents.merge( cohort[["stay_id", "intime"]],on="stay_id",how="inner")

print("Cohort-restricted weight chartevents:", len(chartevents))
print("Weight-event stays:", chartevents["stay_id"].nunique())


In [ ]:
weight_durations_raw =pd.read_csv(DATA / "derived" / "weight_durations.csv", parse_dates=["starttime", "endtime"])


In [ ]:
outputevents["urine_ml"] =pd.to_numeric(outputevents["value"], errors="coerce")
print("urine_ml created, non-null count:", outputevents["urine_ml"].notna().sum())

In [ ]:
#Require same-timestamp pairing of 227488/227489 before applying sign adjustment,to avoid large negative net values from unpaired irrigant entries.
irrigant_pairs =outputevents[outputevents["itemid"].isin([227488, 227489])]
paired_ts =irrigant_pairs.groupby(["stay_id","charttime"])["itemid"].apply(
    lambda x: set(x) =={227488, 227489}
)
paired_keys =set(paired_ts[paired_ts].index)

def is_paired_irrigant(row):
    return (row["stay_id"], row["charttime"]) in paired_keys

outputevents["irrigant_paired"] =outputevents.apply(
    lambda r: is_paired_irrigant(r) if r["itemid"] in [227488, 227489] else True, axis=1
)

#apply the sign adjustment where properly paired only otherwise treat as missing 
outputevents.loc[(outputevents["itemid"]==227488) & (~outputevents["irrigant_paired"]), "urine_ml"] =np.nan

print("urine_ml dtype:", outputevents["urine_ml"].dtype)
print("urine_ml non-null count:", outputevents["urine_ml"].notna().sum())
print("urine_ml null count:", outputevents["urine_ml"].isna().sum())

outputevents =outputevents.dropna(subset=["urine_ml"])

outputevents["urine_ml"] =np.where(
    (outputevents["itemid"] ==227488) & (outputevents["urine_ml"] > 0),
    -1 * outputevents["urine_ml"],
    outputevents["urine_ml"]
)

outputevents["hours_from_icu"] =(outputevents["charttime"] - outputevents["intime"]).dt.total_seconds() / 3600

NON_IRRIGANT_ITEMIDS =[226557, 226558, 226559, 226560, 226561, 226563, 226564, 226565, 226567, 226584]

#Drop negative values for standard urine-collection items (never physiologically valid)
invalid_negative_mask =(outputevents["itemid"].isin(NON_IRRIGANT_ITEMIDS)) & (outputevents["urine_ml"] < 0)
print(f"Dropping {invalid_negative_mask.sum()} invalid negative non-irrigant rows")
outputevents =outputevents[~invalid_negative_mask].copy()
print("outputevents:", outputevents.shape)

In [ ]:
diagnoses =pd.read_csv(HOSP / "diagnoses_icd.csv", usecols=["hadm_id","icd_code","icd_version"])
esrd_hadm_ids =diagnoses[
    ((diagnoses["icd_version"]==9) & (diagnoses["icd_code"]=="5856")) |
    ((diagnoses["icd_version"]==10) & (diagnoses["icd_code"].astype(str).str.startswith("N186", na=False)))
]["hadm_id"].unique()

esrd_stay_ids =set(cohort.loc[cohort["hadm_id"].isin(esrd_hadm_ids), "stay_id"])
print("Cohort stays with ESRD diagnosis:", len(esrd_stay_ids))

dialysis_dependence_hadm_ids =diagnoses[
    ((diagnoses["icd_version"]==9) & (diagnoses["icd_code"].isin(["V451","V4511"]))) |
    ((diagnoses["icd_version"]==10) & (diagnoses["icd_code"]=="Z992"))
]["hadm_id"].unique()
dialysis_stay_ids =set(cohort.loc[cohort["hadm_id"].isin(dialysis_dependence_hadm_ids), "stay_id"])
esrd_dialysis_stay_ids =esrd_stay_ids | dialysis_stay_ids


In [ ]:
CREATININE =50912
creatinine =labevents[labevents["itemid"] ==CREATININE].copy()
creatinine =creatinine.sort_values(["stay_id", "charttime"])
creatinine["hours_from_icu"] =(creatinine["charttime"] - creatinine["intime"]).dt.total_seconds() / 3600
creatinine_orig =creatinine.copy()   #untouched copy; used by validation and the creatinine variants


In [ ]:
crrt_csv =DATA / "derived" / "crrt.csv"
if not crrt_csv.exists():                       #export  from the DuckDB reference tables
    con =duckdb.connect("mimic-iv.duckdb")
    con.execute(f"COPY mimiciv_derived.crrt TO '{crrt_csv}' (HEADER, DELIMITER ',')")
    con.close()
crrt =pd.read_csv(crrt_csv, parse_dates=["charttime"])
crrt_ref_full =crrt.merge(cohort[["stay_id","intime"]], on="stay_id", how="inner")
crrt_ref_full["hours_from_icu"] =(crrt_ref_full["charttime"] - crrt_ref_full["intime"]).dt.total_seconds() / 3600
crrt_active_24_48 =set(crrt_ref_full[(crrt_ref_full["hours_from_icu"]>=24) & (crrt_ref_full["hours_from_icu"]<48)]["stay_id"])


In [ ]:
creatinine_rolling =compute_rolling_creatinine_baseline(creatinine)
creatinine_rolling = creatinine_rolling.merge(
    creatinine[["stay_id", "charttime", "hours_from_icu"]].drop_duplicates(["stay_id", "charttime"]),
    on=["stay_id", "charttime"], how="left")
print(creatinine_rolling.shape)
print(creatinine_rolling[["creat_low_past_48hr", "creat_low_past_7day"]].isna().mean())

kdigo_creatinine_ref = pd.read_csv(DATA / "derived" / "kdigo_creatinine.csv", parse_dates=["charttime"])
chk = creatinine_rolling.merge(
    kdigo_creatinine_ref[["stay_id","charttime","creat_low_past_48hr","creat_low_past_7day"]]
        .drop_duplicates(["stay_id","charttime"]),
    on=["stay_id","charttime"], how="inner", suffixes=("", "_ref"))
for col in ["creat_low_past_48hr", "creat_low_past_7day"]:
    same = (chk[col] == chk[col + "_ref"]) | (chk[col].isna() & chk[col + "_ref"].isna())
    print(f"{col}: {same.mean():.4f} match over {len(chk):,} shared draws")


#### Definitions (variant parameters and functions)

In [ ]:
BASE =dict(max_gap_hours=None, evaluability_mode="exclude",
            baseline_method="fixed", weight_source="current")
def _v(**changes):
    return {**BASE, **changes}

VARIANTS ={
    "baseline": _v(),
    "gap_6h":_v(max_gap_hours=6),
    "gap_12h": _v(max_gap_hours=12),
    "gap_18h":_v(max_gap_hours=18),
    "gap_24h":_v(max_gap_hours=24),
    "unevaluable_negative":_v(evaluability_mode="negative"),
    "unevaluable_excluded_strict": _v(evaluability_mode="strict_exclude_unevaluable"),
    "weight_full_stay": _v(weight_source="full_stay"),
    "weight_durations_ref":_v(weight_source="weight_durations"),
    "rolling_baseline":_v(baseline_method="rolling"),
    "gap12h_rolling":_v(max_gap_hours=12, baseline_method="rolling"),

    #label arithmetic, CRRT and early-exclusion checks
    "exact_arithmetic": _v(exact15=True),
    "crrt_mode_required": _v(crrt_set="mode"),
    "early_excl_crrt":_v(early_crrt=True),
    "early_excl_wd_weight":_v(early_weight="weight_durations"),
    "early_excl_union_ref":_v(early_union_ref=True),

    #weight, baseline and urine-rule variants
    "weight_fill_pre_icu": _v(weight_source="pre_icu_fill"),
    "baseline_median7":_v(baseline_alt="median7"),
    "baseline_ckdepi_fb": _v(baseline_alt="ckdepi_fallback"),
    "baseline_ckdepi_all":_v(baseline_alt="ckdepi_all"),
    "urine_consecutive":_v(urine_method="consecutive"),
}

STABILITY_SET = list(VARIANTS)[:11]      
CHECK_VARIANTS=["exact_arithmetic", "crrt_mode_required", "early_excl_crrt", "early_excl_wd_weight", "early_excl_union_ref"]
EXPLORATORY=["baseline_median7", "baseline_ckdepi_fb", "baseline_ckdepi_all", "urine_consecutive"]


DATA_SWAP=["icu_censored", "baseline_aug", "baseline_gapfix"]



print(len(VARIANTS), "variant parameter sets +", len(DATA_SWAP), "data-swap variants")

In [ ]:
def get_urine_evaluability_labels(aki_labels, mode="exclude"):
    """
    mode="exclude" stays with no baseline AND no urine evaluability are excluded from the cohort upstream. everyone else stays, with urine-unevaluable
        stays defaulting to urine_labels=False.

    mode="negative"same as "exclude", no baseline at all" population is rescued into the cohort as urine-negative instead of dropped.

    mode="strict_exclude_unevaluable" all urine-unevaluable stays are removed from
        the cohort entirely, regardless of creatinine baseline status, testing the opposite"""
    if mode=="exclude":
        return aki_labels
    elif mode=="negative":
        aki_labels = aki_labels.copy()
        aki_labels.loc[~aki_labels["urine_evaluable"], "urine_labels"] = False
        aki_labels["AKI"]= aki_labels["aki_scr"] | aki_labels["urine_labels"]
        return aki_labels
    elif mode== "strict_exclude_unevaluable":
        aki_labels = aki_labels[aki_labels["urine_evaluable"]].copy()
        return aki_labels


In [ ]:
fixed_bl=compute_baseline_scr(creatinine, cohort, method="fixed")[["stay_id", "baseline_scr"]]

pre=creatinine[(creatinine["hours_from_icu"] >= -168) & (creatinine["hours_from_icu"] < 0)]
med7=pre.groupby("stay_id")["valuenum"].median().rename("med7").reset_index()

#CKD-EPI 2021 (race-free), back-calculated creatinine for eGFR = 75 mL/min/1.73 m2
pat = pd.read_csv(HOSP / "patients.csv", usecols=["subject_id", "gender", "anchor_age", "anchor_year"])
ckd_df = cohort[["stay_id", "subject_id", "intime"]].merge(pat, on="subject_id", how="left")
ckd_df["age_raw"] = ckd_df["anchor_age"] + (ckd_df["intime"].dt.year - ckd_df["anchor_year"])
print("missing age:", int(ckd_df["age_raw"].isna().sum()), "| missing gender:", int(ckd_df["gender"].isna().sum()),
      "| age_raw >= 90:", int((ckd_df["age_raw"] >= 90).sum()), "| max age_raw:", ckd_df["age_raw"].max())

#MIMIC deidentifies ages above 89, so age is capped at 90 for the back-calculation
ckd_df["age"] = ckd_df["age_raw"].clip(upper=90)
fem = ckd_df["gender"].eq("F").to_numpy()
kap = np.where(fem, 0.7, 0.9)
ratio = 142*(0.9938 ** ckd_df["age"].to_numpy()) * np.where(fem, 1.012, 1.0) / 75
ok=np.isfinite(ratio) & (ratio > 1)
print("stays without a valid back-calculation:", int((~ok).sum()))
ckd_df["ckd75"]=np.where(ok, (kap * np.where(ok, ratio, 1.0) ** (1 / 1.2)).round(2), np.nan)

bl = (fixed_bl.merge(med7, on="stay_id", how="left")
              .merge(ckd_df[["stay_id", "ckd75"]], on="stay_id", how="left"))
has_pre = bl["med7"].notna()

BASELINE_ALTS = {
    #median instead of minimum of the pre-ICU 7-day values fallback unchanged
    "median7": bl.assign(baseline_scr=bl["med7"].fillna(bl["baseline_scr"]))[["stay_id", "baseline_scr"]],
    #pre-ICU minimum where available. CKD-EPI back-calculation instead of first ICU value
    "ckdepi_fallback": bl.assign(baseline_scr=np.where(has_pre, bl["baseline_scr"], bl["ckd75"]))[["stay_id", "baseline_scr"]],
    #back-calculated baseline for every stay
    "ckdepi_all":bl.assign(baseline_scr=bl["ckd75"])[["stay_id", "baseline_scr"]],}
for k, v in BASELINE_ALTS.items():
    print(k, v["baseline_scr"].describe()[["count", "mean", "50%"]].round(2).to_dict())


In [ ]:

def generate_urine_labels_consecutive(events_df, weight_df, window_start, window_end,max_gap_hours=None, threshold=0.5):
    wmap=weight_df.set_index("stay_id")["weight_kg"]
    rows = []
    for sid, g in events_df.groupby("stay_id"):
        wt =wmap.get(sid, np.nan)
        if not (wt > 0):
            rows.append(dict(stay_id=sid, urine_evaluable_et=False, urine_labels_et=False))
            continue
        g = (g.sort_values("charttime")
              .groupby("charttime", as_index=False)
              .agg(urine_ml=("urine_ml", "sum"), tm=("tm_since_last", "sum"), hrs=("hours_from_icu", "first")))
        t = g["charttime"].values.astype("datetime64[s]").astype("int64")
        ml, tm, hrs = g["urine_ml"].to_numpy(), g["tm"].to_numpy(), g["hrs"].to_numpy()
        with np.errstate(divide="ignore", invalid="ignore"):
            rate = np.where(tm > 0, ml / wt / (tm / 60), np.nan)
        high = (rate >= threshold).astype(int) #NaN counts as not high
        cum_tm, cum_high = np.cumsum(tm), np.cumsum(high)
        evaluable = positive = False
        for i in np.where((hrs >= window_start) & (hrs < window_end))[0]:
            j = np.searchsorted(t, t[i] - 6 * 3600, side="left")
            tm_w = cum_tm[i] - (cum_tm[j - 1] if j > 0 else 0)
            if tm_w / 60 >= 6:
                evaluable = True
                if cum_high[i] - (cum_high[j - 1] if j > 0 else 0) == 0:
                    positive = True
                    break
        rows.append(dict(stay_id=sid, urine_evaluable_et=evaluable, urine_labels_et=positive))
    return pd.DataFrame(rows)

In [ ]:
def get_scr_labels(baseline_method, baseline_scr, creatinine_window_df, cohort_ids_df, outcome_start, outcome_end):
    """
    Returns (aki_1_5x_df, aki_0_3_df) for the given window under the chosen baseline method. creatinine_window_df is only used by the 'fixed' method.
    """
    if baseline_method=="fixed":
        if outcome_start==0:
            scr_win = get_time_window(creatinine, 0, 24)
            return kdigo_scr_15x(scr_win, baseline_scr), kdigo_scr_03_early(scr_win)
        else:
            scr_win = get_time_window(creatinine, outcome_start, outcome_end).merge(
                cohort_ids_df[["stay_id"]], on="stay_id", how="inner")
            creat_outcome = creatinine.merge(cohort_ids_df[["stay_id"]], on="stay_id", how="inner")
            return (kdigo_scr_15x(scr_win, baseline_scr),
                    kdigo_scr_03(creat_outcome, outcome_start=outcome_start, outcome_end=outcome_end))

    elif baseline_method == "rolling":
        roll = creatinine_rolling.merge(cohort_ids_df[["stay_id"]], on="stay_id", how="inner")
        return (kdigo_scr_15x_rolling(roll, outcome_start=outcome_start, outcome_end=outcome_end),
                kdigo_scr_03_rolling(roll, outcome_start=outcome_start, outcome_end=outcome_end))

    


In [ ]:
kdigo_stages_ref=pd.read_csv(DATA / "derived" / "kdigo_stages.csv", parse_dates=["charttime"])
ref = kdigo_stages_ref.merge(cohort[["stay_id","intime"]], on="stay_id", how="inner")
ref["hours_from_icu"]=(ref["charttime"] - ref["intime"]).dt.total_seconds() / 3600
ref_24_48 = ref[(ref["hours_from_icu"] >= 24) & (ref["hours_from_icu"] < 48)]

ref_early_ids = set(ref.loc[(ref["hours_from_icu"] >= 0) & (ref["hours_from_icu"] < 24) & (ref["aki_stage"] > 0), "stay_id"])
crrt_early_ids = set(crrt_ref_full[(crrt_ref_full["hours_from_icu"] >= 0) & (crrt_ref_full["hours_from_icu"] < 24)]["stay_id"])
crrt_mode_24_48 = set(crrt_ref_full[crrt_ref_full["crrt_mode"].notna()
                      & (crrt_ref_full["hours_from_icu"] >= 24) & (crrt_ref_full["hours_from_icu"] < 48)]["stay_id"])



In [ ]:
def exact15_ids(win, baseline_scr):
    d = win.merge(baseline_scr[["stay_id", "baseline_scr"]], on="stay_id").dropna(subset=["baseline_scr", "valuenum"])
    v = np.rint(d["valuenum"] * 100).astype(int); b = np.rint(d["baseline_scr"] * 100).astype(int)
    return set(d.loc[2 * v >= 3 * b, "stay_id"]) #Integer test of value >= 1.5 x baseline

def add_exact15(scr_labels, win, baseline_scr):
    ids = exact15_ids(win, baseline_scr)
    scr_labels = scr_labels.copy()
    scr_labels["aki_1_5x"] = scr_labels["aki_1_5x"].fillna(False).astype(bool) | scr_labels["stay_id"].isin(ids)
    scr_labels["aki_scr"] = scr_labels["aki_1_5x"] | scr_labels["aki_0_3"].fillna(False).astype(bool)
    return scr_labels

In [ ]:
def weight_with_pre_icu_fill(chartevents):
    """v5 first-24h weight for stays without one, the latest pre-ICU weight (kg items preferred)."""
    base_w = compute_weight_table(chartevents, method="first_24h")[["stay_id", "weight_kg"]]
    w = chartevents.copy()
    w["h"] = (w["charttime"] - w["intime"]).dt.total_seconds() / 3600
    w = w[(w["h"] < 0) & ~w["stay_id"].isin(base_w["stay_id"])]
    v = pd.to_numeric(w["valuenum"], errors="coerce")
    w["weight_kg"] = np.where(w["itemid"] == 226531, v * 0.45359237, v)
    w = w[w["weight_kg"].between(20, 300)]
    w["priority"] = np.where(w["itemid"].isin([226512, 224639]), 0, 1)
    w = w.sort_values(["stay_id", "priority", "charttime"], ascending=[True, True, False])
    fill = w.groupby("stay_id", as_index=False).first()[["stay_id", "weight_kg"]]
    return pd.concat([base_w, fill], ignore_index=True)


In [ ]:
def run_variant(variant_name, params):
    if params["weight_source"] == "current":
        weight = compute_weight_table(chartevents, method="first_24h")
    elif params["weight_source"] == "full_stay":
        weight = compute_weight_table(chartevents, method="full_stay_last_known")
    elif params["weight_source"] == "weight_durations":
        weight = compute_weight_from_durations(weight_durations_raw, cohort)
    elif params["weight_source"] == "pre_icu_fill":
        weight = weight_with_pre_icu_fill(chartevents)

    early_weight = (compute_weight_from_durations(weight_durations_raw, cohort)
                    if params.get("early_weight") == "weight_durations" else weight)

    baseline_scr = compute_baseline_scr(creatinine, cohort, method="fixed")
    label_baseline = BASELINE_ALTS.get(params.get("baseline_alt"), baseline_scr)
    urine_fn = (generate_urine_labels_consecutive if params.get("urine_method") == "consecutive"
                else generate_urine_labels_elapsed_time)

    early_scr_15x, early_scr_03 = get_scr_labels(params["baseline_method"], label_baseline, creatinine, cohort, 0, 24)
    early_scr_labels = combine_scr_labels(cohort, early_scr_15x, early_scr_03)
    if params.get("exact15"):
        early_scr_labels = add_exact15(early_scr_labels, get_time_window(creatinine, 0, 24), baseline_scr)

    early_urine_labels_et = urine_fn(
        outputevents[outputevents["stay_id"].isin(cohort["stay_id"])],
        early_weight, window_start=0, window_end=24, max_gap_hours=params["max_gap_hours"])
    early_urine_labels = early_urine_labels_et.rename(columns={"urine_labels_et": "urine_labels"})[["stay_id", "urine_labels"]]

    early_kdigo = early_scr_labels.merge(early_urine_labels, on="stay_id", how="left")
    early_kdigo["urine_labels"] = early_kdigo["urine_labels"].fillna(False)
    early_kdigo["early_AKI"] = early_kdigo["aki_scr"] | early_kdigo["urine_labels"]
    if params.get("early_crrt"):
        early_kdigo["early_AKI"] |= early_kdigo["stay_id"].isin(crrt_early_ids)
    if params.get("early_union_ref"):
        early_kdigo["early_AKI"] |= early_kdigo["stay_id"].isin(ref_early_ids)
    early_aki_ids = set(early_kdigo.loc[early_kdigo["early_AKI"], "stay_id"])

    eligible_cohort = cohort.merge(early_kdigo[["stay_id", "early_AKI"]], on="stay_id", how="left")
    eligible_cohort["early_AKI"] = eligible_cohort["early_AKI"].fillna(False)
    eligible_cohort = eligible_cohort[~eligible_cohort["early_AKI"]].copy()

    scr_15x_labels, scr_03_labels = get_scr_labels(params["baseline_method"], label_baseline, creatinine, eligible_cohort, 24, 48)
    scr_labels = combine_scr_labels(eligible_cohort, scr_15x_labels, scr_03_labels)
    if params.get("exact15"):
        w_out = get_time_window(creatinine, 24, 48).merge(eligible_cohort[["stay_id"]], on="stay_id")
        scr_labels = add_exact15(scr_labels, w_out, baseline_scr)

    urine_labels_et = urine_fn(
        outputevents[outputevents["stay_id"].isin(eligible_cohort["stay_id"])],
        weight, window_start=24, window_end=48, max_gap_hours=params["max_gap_hours"])
    urine_evaluable_ids = set(urine_labels_et.loc[urine_labels_et["urine_evaluable_et"], "stay_id"])
    no_scr = set(baseline_scr.loc[baseline_scr["baseline_scr"].isna(), "stay_id"])

    if params["evaluability_mode"] == "exclude":
        eligible_cohort = eligible_cohort[~eligible_cohort["stay_id"].isin(no_scr - urine_evaluable_ids)].copy()
    elif params["evaluability_mode"] == "negative":
        no_urine_data_at_all = set(cohort["stay_id"]) - set(outputevents["stay_id"].unique())
        eligible_cohort = eligible_cohort[~eligible_cohort["stay_id"].isin(no_scr & no_urine_data_at_all)].copy()
    elif params["evaluability_mode"] == "strict_exclude_unevaluable":
        pass
    

    scr_labels = scr_labels[scr_labels["stay_id"].isin(eligible_cohort["stay_id"])].copy()
    aki_labels = scr_labels.merge(urine_labels_et, on="stay_id", how="left")
    aki_labels["urine_evaluable"] = aki_labels["urine_evaluable_et"].fillna(False)
    aki_labels["urine_labels"] = aki_labels["urine_labels_et"].fillna(False)
    aki_labels["AKI"] = aki_labels["aki_scr"] | aki_labels["urine_labels"]
    aki_labels = get_urine_evaluability_labels(aki_labels, mode=params["evaluability_mode"])
    eligible_cohort = eligible_cohort[eligible_cohort["stay_id"].isin(aki_labels["stay_id"])].copy()

    crrt_set = crrt_mode_24_48 if params.get("crrt_set") == "mode" else crrt_active_24_48
    aki_labels["on_crrt"] = aki_labels["stay_id"].isin(crrt_set)
    aki_labels["AKI"] = aki_labels["AKI"] | aki_labels["on_crrt"]

    aki_labels.to_parquet(VARIANT_OUTPUT / f"{variant_name}_labels.parquet", index=False)
    print(f"Variant '{variant_name}': cohort={len(eligible_cohort)}, AKI positive={aki_labels['AKI'].sum()}, "
          f"prevalence={aki_labels['AKI'].mean()*100:.2f}%")
    return aki_labels, eligible_cohort, early_aki_ids


#### Load saved results

In [ ]:
RERUN = False   # False: use the saved pkl if it exists.  True: recompute all 24 variants
SEED, N_BOOT = 42, 1000
CACHE = OUTPUT / "Sensitivity_Analysis" / "all_results_cache.pkl"
PARTIAL = OUTPUT / "Sensitivity_Analysis" / "all_results_partial.pkl"

if CACHE.exists() and not RERUN:
    with open(CACHE, "rb") as f:
        all_results = pickle.load(f)
    old_results = None
    print("Loaded", len(all_results), "variants from cache; nothing recomputed")
else:
    old_results = pickle.load(open(CACHE, "rb")) if CACHE.exists() else None
    all_results = pickle.load(open(PARTIAL, "rb")) if PARTIAL.exists() else {}
    def _save_partial():
        with open(PARTIAL, "wb") as f:
            pickle.dump(all_results, f, protocol=pickle.HIGHEST_PROTOCOL)

    # 21 parameter sets
    for name, params in VARIANTS.items():
        if name in all_results:
            print("skip (already done):", name); continue
        all_results[name] = run_variant(name, params); _save_partial()

    # data swaps: swap one global table, run, always restore
    if "icu_censored" not in all_results:
        cens = creatinine_orig.merge(cohort[["stay_id", "outtime"]], on="stay_id")
        try:
            creatinine = cens[cens["charttime"] <= cens["outtime"]].drop(columns=["outtime"])
            all_results["icu_censored"] = run_variant("icu_censored", VARIANTS["baseline"])
        finally:
            creatinine = creatinine_orig
        _save_partial()

    if "baseline_aug" not in all_results:
        cr_raw = labevents_raw[(labevents_raw["itemid"] == 50912) & labevents_raw["subject_id"].isin(cohort["subject_id"])]
        cand = cr_raw.merge(cohort[["subject_id", "stay_id", "intime", "outtime", "hadm_id"]]
                            .rename(columns={"hadm_id": "cohort_hadm"}), on="subject_id")
        cand["h_from_icu"] = (cand["charttime"] - cand["intime"]).dt.total_seconds() / 3600
        cand = cand[(cand["h_from_icu"] >= -168) & (cand["charttime"] <= cand["outtime"])]
        have = creatinine_orig[["stay_id", "charttime"]].drop_duplicates().assign(have=True)
        miss = cand.merge(have, on=["stay_id", "charttime"], how="left")
        miss = miss[miss["have"].isna()].copy()
        extra = miss[["subject_id", "hadm_id", "itemid", "charttime", "valuenum", "stay_id", "intime"]]
        creatinine_aug = (pd.concat([creatinine_orig.drop(columns=["hours_from_icu"]), extra], ignore_index=True)
                          .sort_values(["stay_id", "charttime"]))
        creatinine_aug["hours_from_icu"] = (creatinine_aug["charttime"] - creatinine_aug["intime"]).dt.total_seconds() / 3600
        try:
            creatinine = creatinine_aug
            all_results["baseline_aug"] = run_variant("baseline_aug", VARIANTS["baseline"])
        finally:
            creatinine = creatinine_orig
        _save_partial()

    if "baseline_gapfix" not in all_results:
        outputevents_orig = outputevents.copy()
        try:
            outputevents = compute_measurement_gaps(outputevents_orig.drop(columns=["tm_since_last"]))
            all_results["baseline_gapfix"] = run_variant("baseline_gapfix", VARIANTS["baseline"])
        finally:
            outputevents = outputevents_orig
        _save_partial()

    assert len(all_results) == 24
    with open(CACHE, "wb") as f:
        pickle.dump(all_results, f, protocol=pickle.HIGHEST_PROTOCOL)
    PARTIAL.unlink(missing_ok=True)

assert len(all_results) == 24
for k in EXPLORATORY + ["baseline_gapfix", "icu_censored", "baseline_aug"]:
    assert k in all_results, f"{k} missing"

#### Reference tables (mimiciv_derived) and  helpers

In [ ]:
if old_results is not None: # only after a real rerun
    rec = []
    for name in all_results:
        nl, nc, ne = all_results[name]; ol, oc, oe = old_results[name]
        m = nl.merge(ol, on="stay_id", suffixes=("_n", "_o"))
        rec.append({"variant": name,
                    "cohort_equal": set(nl.stay_id) == set(ol.stay_id),
                    "early_excluded_equal": set(ne) == set(oe),
                    "label_flips_vs_old_run": int((m["AKI_n"].astype(bool) != m["AKI_o"].astype(bool)).sum())})
    recon = pd.DataFrame(rec); print(recon.to_string(index=False))
    recon.to_csv(OUTPUT / "reconciliation_old_vs_rerun.csv", index=False)
    assert recon["cohort_equal"].all() and recon["early_excluded_equal"].all() and (recon["label_flips_vs_old_run"] == 0).all()

fz = pd.read_parquet(OUTPUT / "labels_v5.parquet") # always runs
b = all_results["baseline"][0]
assert len(b) == len(fz) == 28977 and int(b["AKI"].sum()) == 7624
m = fz.merge(b, on="stay_id", suffixes=("_f", "_b"))
for c in ["aki_0_3", "aki_1_5x", "urine_labels", "aki_scr", "urine_evaluable", "on_crrt", "AKI"]:
    assert (m[c + "_f"].astype(bool) == m[c + "_b"].astype(bool)).all(), c
print("Baseline equals frozen labels_v5")

In [ ]:
weight = compute_weight_table(chartevents, method="first_24h")   #v5 first-24h weight rule, identical to run_variant
print("stays with a usable first-24h weight:", weight["stay_id"].nunique())

In [ ]:
ref_label = ref_24_48.groupby("stay_id")["aki_stage"].max().reset_index().assign(AKI_ref=lambda d: d["aki_stage"] > 0)

ref_label_creat = (ref_24_48.groupby("stay_id")["aki_stage_creat"].max()
                   .reset_index().assign(AKI_ref_creat=lambda d: d["aki_stage_creat"] > 0))

ref_label_uo = (ref_24_48.groupby("stay_id")["aki_stage_uo"].max().reset_index()
                .assign(AKI_ref_uo=lambda d: d["aki_stage_uo"] > 0))
refs = (ref_label[["stay_id","AKI_ref"]]
        .merge(ref_label_creat[["stay_id","AKI_ref_creat"]], on="stay_id", how="left")
        .merge(ref_label_uo[["stay_id","AKI_ref_uo"]], on="stay_id", how="left").fillna(False))

def kappa_np(y, p):
    a = (y & p).sum(); b = (~y & p).sum(); c = (y & ~p).sum(); d = (~y & ~p).sum()
    n = a + b + c + d; po = (a + d) / n; pe = ((a + b) * (a + c) + (c + d) * (b + d)) / n**2
    return (po - pe) / (1 - pe)

def kap(d, p_col, r_col):
    return round(kappa_np(d[r_col].to_numpy(bool), d[p_col].to_numpy(bool)), 4)

OUTCOMES = {"combined": ("AKI", "AKI_ref"), "creatinine": ("aki_scr", "AKI_ref_creat"), "urine": ("urine_labels", "AKI_ref_uo")}

#### Comparison of all variants against mimiciv_derived

In [ ]:

def summarize_variant(name, aki_labels, eligible_cohort, early_aki_ids):
    compare = aki_labels.merge(ref_label[["stay_id","AKI_ref"]], on="stay_id", how="inner")
    kappa = cohen_kappa_score(compare["AKI"], compare["AKI_ref"]) if len(compare) else float("nan")
    return {
        "Variant": name,
        "Cohort N": eligible_cohort["stay_id"].nunique(),
        "Early-Excluded N": len(early_aki_ids),
        "Evaluable N": int(aki_labels["urine_evaluable"].sum()),
        "AKI N": int(aki_labels["AKI"].sum()),
        "AKI %": round(aki_labels["AKI"].mean()*100, 2),
        "Kappa": round(kappa, 4)
    }

comparison_rows = [
    summarize_variant(name, aki_labels, eligible_cohort, early_aki_ids)
    for name, (aki_labels, eligible_cohort, early_aki_ids) in all_results.items()
]
comparison_table = pd.DataFrame(comparison_rows)
print(comparison_table)
comparison_table.to_csv(OUTPUT / "phase2_comparison_all_variants.csv", index=False)


#### Gap sweep

In [ ]:

gap_variant_names = ["baseline", "gap_6h", "gap_12h", "gap_18h", "gap_24h"]
gap_sweep_table = comparison_table[comparison_table["Variant"].isin(gap_variant_names)].copy()
gap_order = {"gap_6h": 6, "gap_12h": 12, "gap_18h": 18, "gap_24h": 24, "baseline": 999}
gap_sweep_table["sort_key"] = gap_sweep_table["Variant"].map(gap_order)
gap_sweep_table = gap_sweep_table.sort_values("sort_key").drop(columns="sort_key")
print(gap_sweep_table)
gap_sweep_table.to_csv(OUTPUT / "gap_limit_sweep_table.csv", index=False)

In [ ]:
def full_stats_variant(aki_labels, ref_label, label):
    compare = aki_labels.merge(ref_label[["stay_id","AKI_ref"]], on="stay_id", how="inner")
    tn, fp, fn, tp = confusion_matrix(compare["AKI_ref"], compare["AKI"]).ravel()
    sens = tp/(tp + fn) if (tp+fn) else float('nan')
    spec = tn/(tn + fp) if (tn+fp) else float('nan')
    ppv = tp/(tp + fp) if (tp+fp) else float('nan')
    npv = tn/(tn + fn) if (tn+fn) else float('nan')
    kappa = cohen_kappa_score(compare["AKI_ref"], compare["AKI"])
    print(f"{label}")
    print(f"TP={tp} FP={fp} FN={fn} TN={tn}")
    print(f"Sensitivity: {sens:.4f}  Specificity: {spec:.4f}  PPV: {ppv:.4f}  NPV: {npv:.4f}  Kappa: {kappa:.4f}")
    print()
    return {"label": label, "TP": tp, "FP": fp, "FN": fn, "TN": tn,
            "sensitivity": sens, "specificity": spec, "ppv": ppv, "npv": npv, "kappa": kappa}

gap_variant_names = ["baseline", "gap_6h", "gap_12h", "gap_18h", "gap_24h"]
gap_stats_rows = []
for name in gap_variant_names:
    aki_labels_v, _, _ = all_results[name]
    gap_stats_rows.append(full_stats_variant(aki_labels_v, ref_label, name))

gap_stats_df = pd.DataFrame(gap_stats_rows)
gap_stats_df.to_csv(OUTPUT / "gap_sweep_full_stats.csv", index=False)
print(gap_stats_df)

#### Per-stay robustness

In [ ]:
all_labels_matrix = pd.DataFrame({n: all_results[n][0].set_index("stay_id")["AKI"] for n in STABILITY_SET})
all_labels_matrix["n_variants_present"] = all_labels_matrix.notna().sum(axis=1)
all_labels_matrix["n_variants_positive"] = all_labels_matrix.drop(columns=["n_variants_present"]).sum(axis=1, skipna=True)
all_labels_matrix["fraction_positive"] = all_labels_matrix["n_variants_positive"] / all_labels_matrix["n_variants_present"]
print(all_labels_matrix["fraction_positive"].describe())
robust_negative = (all_labels_matrix["fraction_positive"] == 0).sum()
robust_positive = (all_labels_matrix["fraction_positive"] == 1).sum()
fragile = len(all_labels_matrix) - robust_negative - robust_positive
N = len(all_labels_matrix)
print(f"stays in matrix: {N}")
print(f"Robust negative: {robust_negative} ({robust_negative/N*100:.1f}%)")
print(f"Robust positive: {robust_positive} ({robust_positive/N*100:.1f}%)")
print(f"Fragile: {fragile} ({fragile/N*100:.1f}%)")
all_labels_matrix.to_csv(OUTPUT / "robustness_score_matrix.csv")



In [ ]:
base_ids = set(all_results["baseline"][1]["stay_id"])
roll_ids = set(all_results["rolling_baseline"][1]["stay_id"])
dropped = base_ids - roll_ids
base_lab = all_results["baseline"][0]
d = base_lab[base_lab["stay_id"].isin(dropped)]
print(len(dropped), "stays dropped; baseline AKI rate among them:", round(d["AKI"].mean()*100, 2), "%")
print("baseline AKI rate among the rest:", round(base_lab[~base_lab["stay_id"].isin(dropped)]["AKI"].mean()*100, 2), "%")

#### Short-LOS interaction, creatinine-only kappa, common-stay comparison

In [ ]:

short_stay_ids = set(cohort.loc[cohort["los_hours"] < 48, "stay_id"])
print("Short-stay (LOS<48h) stay_ids captured:", len(short_stay_ids))
for name in gap_variant_names:
    aki_labels_v, eligible_cohort_v, _ = all_results[name]
    aki_labels_v = aki_labels_v.copy()
    aki_labels_v["is_short_los"] = aki_labels_v["stay_id"].isin(short_stay_ids)

    short_prev = aki_labels_v.loc[aki_labels_v["is_short_los"], "AKI"].mean() * 100
    long_prev = aki_labels_v.loc[~aki_labels_v["is_short_los"], "AKI"].mean() * 100
    short_n = aki_labels_v["is_short_los"].sum()
    long_n = (~aki_labels_v["is_short_los"]).sum()

    print(f"{name}: short-LOS AKI%={short_prev:.2f} (n={short_n}), long-LOS AKI%={long_prev:.2f} (n={long_n}), "
          f"gap={short_prev - long_prev:+.2f}")




In [ ]:

for name in ["baseline", "rolling_baseline", "gap12h_rolling"]:
    lab = all_results[name][0]
    c = lab.merge(ref_label_creat[["stay_id", "AKI_ref_creat"]], on="stay_id", how="inner")
    print(name, "creatinine-only kappa:", round(cohen_kappa_score(c["AKI_ref_creat"], c["aki_scr"]), 4), "n =", len(c))

In [ ]:
common = set(all_results["baseline"][0]["stay_id"]) & set(all_results["rolling_baseline"][0]["stay_id"])
for name in ["baseline", "rolling_baseline"]:
    lab = all_results[name][0]
    lab = lab[lab["stay_id"].isin(common)]
    c = lab.merge(ref_label[["stay_id","AKI_ref"]], on="stay_id").merge(
        ref_label_creat[["stay_id","AKI_ref_creat"]], on="stay_id")
    print(name, "n =", len(c),
          "| combined kappa:", round(cohen_kappa_score(c["AKI_ref"], c["AKI"]), 4),
          "| creatinine-only kappa:", round(cohen_kappa_score(c["AKI_ref_creat"], c["aki_scr"]), 4))

#### Evaluable-urine subset

In [ ]:
b = all_results["baseline"][0]
s = all_results["unevaluable_excluded_strict"][0]
strict_ids = set(s["stay_id"])

def kap(d, p_col, r_col):
    return round(kappa_np(d[r_col].to_numpy(bool), d[p_col].to_numpy(bool)), 4)

bm = b.merge(refs, on="stay_id", how="inner")
sm = s.merge(refs, on="stay_id", how="inner")

pops = {
    "baseline, full cohort": bm,
    "baseline, evaluable-urine stays only": bm[bm["stay_id"].isin(strict_ids)],
    "baseline, unevaluable-urine stays only":bm[~bm["stay_id"].isin(strict_ids)],
    "strict-exclusion variant (own cohort)": sm,
}
rows = []
for label, d in pops.items():
    rows.append({
        "Population": label,
        "n compared": len(d),
        "AKI % (ours)": round(d["AKI"].mean() * 100, 2),
        "AKI % (mimiciv_derived)": round(d["AKI_ref"].mean() * 100, 2),
        "Kappa combined": kap(d, "AKI", "AKI_ref"),
        "Kappa creatinine": kap(d, "aki_scr", "AKI_ref_creat"),
        "Kappa urine": kap(d, "urine_labels", "AKI_ref_uo"),
    })
evaluable_cmp = pd.DataFrame(rows)
print(evaluable_cmp.to_string(index=False))
evaluable_cmp.to_csv(OUTPUT / "evaluable_subset_kappa.csv", index=False)

#### table and paired bootstrap (stay level)

In [ ]:
SECTION3 = ["baseline", "gap_6h", "gap_12h", "unevaluable_negative", "unevaluable_excluded_strict",
            "weight_full_stay", "weight_durations_ref", "rolling_baseline", "gap12h_rolling",
            "baseline_aug", "icu_censored"]
base_lab = all_results["baseline"][0]
base_ids = set(base_lab["stay_id"])
base_AKI = base_lab.set_index("stay_id")["AKI"]
ref_small = ref_label[["stay_id", "AKI_ref"]]

rows = []
for name in SECTION3:
    lab, coh, early_ids = all_results[name]
    v = lab.set_index("stay_id")["AKI"]
    common = sorted(base_ids & set(v.index))
    changed = int((v.loc[common] != base_AKI.loc[common]).sum())
    c_all = lab.merge(ref_small, on="stay_id")
    c_com = lab[lab["stay_id"].isin(base_ids)].merge(ref_small, on="stay_id")
    rows.append({"Variant": name, "Cohort N": coh["stay_id"].nunique(),
                 "Early-Excluded N": len(early_ids), "Evaluable N": int(lab["urine_evaluable"].sum()),
                 "AKI N": int(lab["AKI"].sum()), "AKI %": round(lab["AKI"].mean() * 100, 2),
                 "Common w/ baseline N": len(common), "Labels Changed": changed,
                 "Changed %": round(changed / len(common) * 100, 2),
                 "n compared": len(c_all), "Kappa vs mimiciv_derived": round(cohen_kappa_score(c_all["AKI_ref"], c_all["AKI"]), 4),
                 "n compared (common)": len(c_com), "Kappa (common stays)": round(cohen_kappa_score(c_com["AKI_ref"], c_com["AKI"]), 4)})
section3 = pd.DataFrame(rows)
print(section3.to_string(index=False))
section3.to_csv(OUTPUT / "paper1_section3_table.csv", index=False)





In [ ]:

w_base = compute_weight_table(chartevents, method="first_24h"); w_fill = weight_with_pre_icu_fill(chartevents)
print("stays with a weight - v5 rule:", w_base.stay_id.nunique(), "| with pre-ICU fill:", w_fill.stay_id.nunique(),
      "| newly filled (in cohort):", len((set(w_fill.stay_id) - set(w_base.stay_id)) & cohort_stay_ids))

lab, coh, early = all_results["weight_fill_pre_icu"]
d = lab.merge(refs, on="stay_id")
print("cohort", len(coh), "| early-excl", len(early), "| urine-evaluable", int(lab["urine_evaluable"].sum()),
      "| AKI", int(lab["AKI"].sum()), f"({lab['AKI'].mean()*100:.2f}%) | n compared", len(d))
print("abs kappa  combined", round(kappa_np(d["AKI_ref"].to_numpy(bool), d["AKI"].to_numpy(bool)), 4),
      "| creat", round(kappa_np(d["AKI_ref_creat"].to_numpy(bool), d["aki_scr"].to_numpy(bool)), 4),
      "| urine", round(kappa_np(d["AKI_ref_uo"].to_numpy(bool), d["urine_labels"].to_numpy(bool)), 4))
v = lab.set_index("stay_id")["AKI"]; b = all_results["baseline"][0].set_index("stay_id")["AKI"]
common = sorted(set(v.index) & set(b.index))
print("labels changed vs v5 on", len(common), "common stays:", int((v.loc[common] != b.loc[common]).sum()))






In [ ]:
base = all_results["baseline"][0]
lab = all_results["weight_fill_pre_icu"][0]
v = lab.set_index("stay_id"); b = base.set_index("stay_id")
common = sorted(set(v.index) & set(b.index))

In [ ]:

for c in ["urine_labels", "urine_evaluable", "aki_scr", "on_crrt"]:
    print(c, "changed:", int((v.loc[common, c].astype(bool) != b.loc[common, c].astype(bool)).sum()))

#stays dropped from the cohort by the extra early exclusion
dropped = set(base["stay_id"]) - set(lab["stay_id"])
print("\nbaseline stays not in variant cohort:", len(dropped),
      "| of those v5-AKI positive:", int(b.loc[list(dropped), "AKI"].astype(bool).sum()))


In [ ]:
def paired_dk(lab, pc, rc, B=N_BOOT, seed=SEED):
    d = (all_results["baseline"][0][["stay_id", pc]]
         .merge(lab[["stay_id", pc]], on="stay_id", suffixes=("_b", "_v"))
         .merge(refs[["stay_id", rc]], on="stay_id"))
    y = d[rc].to_numpy(bool); pb = d[pc+"_b"].to_numpy(bool); pv = d[pc+"_v"].to_numpy(bool)
    obs = kappa_np(y, pv) - kappa_np(y, pb)
    rng = np.random.default_rng(seed); bs = np.empty(B)
    for k in range(B):
        i = rng.integers(0, len(d), len(d))
        bs[k] = kappa_np(y[i], pv[i]) - kappa_np(y[i], pb[i])
    lo, hi = np.percentile(bs, [2.5, 97.5])
    return round(obs, 4), round(lo, 4), round(hi, 4), len(d)

In [ ]:
base_AKI = all_results["baseline"][0].set_index("stay_id")["AKI"]
rows = []
for name, (lab, coh, early) in all_results.items():
    if name == "baseline":
        continue
    v = lab.set_index("stay_id")["AKI"]
    common = sorted(set(base_AKI.index) & set(v.index))
    row = {
           "Variant": name,
           "Cohort N": coh["stay_id"].nunique(), "Early-excl. N": len(early),
           "AKI N": int(lab["AKI"].sum()), "AKI %": round(lab["AKI"].mean()*100, 2),
           "Common N": len(common), "Labels changed": int((v.loc[common] != base_AKI.loc[common]).sum())}
    for oname, (pc, rc) in OUTCOMES.items():
        o, lo, hi, n = paired_dk(lab, pc, rc)
        row.update({f"dkappa {oname}": o, f"{oname} CI low": lo, f"{oname} CI high": hi, f"{oname} n": n})
    rows.append(row)
paired_all = pd.DataFrame(rows)
paired_all.to_csv(OUTPUT / "paired_dkappa_all_variants.csv", index=False)
print(paired_all.to_string(index=False))

#### Consecutive-interval urine rule and baseline definitions

In [ ]:
base_pos = set(all_results["baseline"][0].query("urine_labels").stay_id)
cons_pos = set(all_results["urine_consecutive"][0].query("urine_labels").stay_id)
print("consecutive positives:", len(cons_pos), "| not in elapsed positives:", len(cons_pos - base_pos))




#### Urine kappa per variant and weight availability

In [ ]:

for name, (lab, _, _) in all_results.items():
    d = lab.merge(refs[["stay_id","AKI_ref_uo"]], on="stay_id")
    print(f"{name:28s} urine kappa {cohen_kappa_score(d['AKI_ref_uo'], d['urine_labels']):.4f}  n={len(d)}")

has_w = set(weight["stay_id"])
d = all_results["baseline"][0].merge(refs[["stay_id","AKI_ref_uo"]], on="stay_id")
for flag, g in d.groupby(d["stay_id"].isin(has_w)):
    print("weight available" if flag else "no weight      ", len(g),
          "| urine kappa", round(cohen_kappa_score(g["AKI_ref_uo"], g["urine_labels"]), 4),
          "| reference-positive:", int(g["AKI_ref_uo"].sum()))


#### Summary table


In [ ]:

base_lab = all_results["baseline"][0]
base_AKI = base_lab.set_index("stay_id")["AKI"]


rows = []
for name, (lab, coh, early_ids) in all_results.items():
    d = lab.merge(refs, on="stay_id", how="inner")
    v = lab.set_index("stay_id")["AKI"]
    common = sorted(set(base_lab["stay_id"]) & set(v.index))
    changed = int((v.loc[common] != base_AKI.loc[common]).sum())
    rows.append({
        "Variant": name,
        "Cohort N": coh["stay_id"].nunique(),
        "Early-Excluded N": len(early_ids),
        "Evaluable N": int(lab["urine_evaluable"].sum()),
        "AKI N": int(lab["AKI"].sum()),
        "AKI %": round(lab["AKI"].mean() * 100, 2),
        "n compared": len(d),
        "Kappa combined": kap(d, "AKI", "AKI_ref"),
        "Kappa creatinine": kap(d, "aki_scr", "AKI_ref_creat"),
        "Kappa urine": kap(d, "urine_labels", "AKI_ref_uo"),
        "Common w/ baseline N": len(common),
        "Labels Changed": changed,
        "Changed %": round(changed / len(common) * 100, 2),
    })
phase2_final = pd.DataFrame(rows)
print(phase2_final.to_string(index=False))
phase2_final.to_csv(OUTPUT / "phase2_final_summary.csv", index=False)



In [ ]:
DESIGN = {   # name: (changed parameter, affects by design)
 "baseline": ("none (primary V5)", "reference"),
 "gap_6h": ("max_gap_hours=6", "both"), "gap_12h": ("max_gap_hours=12", "both"),
 "gap_18h": ("max_gap_hours=18", "both"), "gap_24h": ("max_gap_hours=24", "both"),
 "unevaluable_negative": ("evaluability_mode=negative", "cohort"),
 "unevaluable_excluded_strict": ("evaluability_mode=strict_exclude_unevaluable", "cohort"),
 "weight_full_stay": ("weight_source=full_stay", "both"),
 "weight_durations_ref": ("weight_source=weight_durations", "both"),
 "rolling_baseline": ("baseline_method=rolling", "both"),
 "gap12h_rolling": ("max_gap_hours=12 + baseline_method=rolling", "both"),
 "exact_arithmetic": ("exact15=True (integer 1.5x test)", "both"),
 "crrt_mode_required": ("crrt_set=mode", "labels"),
 "early_excl_crrt": ("early_crrt=True", "cohort"),
 "early_excl_wd_weight": ("early_weight=weight_durations", "cohort"),
 "early_excl_union_ref": ("early_union_ref=True", "cohort"),
 "weight_fill_pre_icu": ("weight_source=pre_icu_fill", "both"),
 "baseline_median7": ("baseline_alt=median7", "both"),
 "baseline_ckdepi_fb": ("baseline_alt=ckdepi_fallback", "both"),
 "baseline_ckdepi_all": ("baseline_alt=ckdepi_all", "both"),
 "urine_consecutive": ("urine_method=consecutive", "both"),
 "icu_censored": ("data swap: creatinine after ICU outtime removed", "both"),
 "baseline_aug": ("data swap: extra creatinine rows added", "both"),
 "baseline_gapfix": ("data swap: urine gaps recomputed after cleaning", "both"),
}
base_lab, base_coh, base_early = all_results["baseline"]
base_ids = set(base_lab["stay_id"]); base_AKI = base_lab.set_index("stay_id")["AKI"].astype(bool)
ref_ids = set(ref_label["stay_id"])
rows = []
for name, (lab, coh, early) in all_results.items():
    ids = set(lab["stay_id"]); v = lab.set_index("stay_id")["AKI"].astype(bool)
    common = sorted(base_ids & ids); left = list(base_ids - ids); entered = list(ids - base_ids)
    b, x = base_AKI.loc[common], v.loc[common]
    cohort_chg = (len(left) + len(entered)) > 0; label_chg = int((b != x).sum()) > 0
    rows.append({
                "variant": name,
                 "changed_parameter": DESIGN[name][0], "affects_by_design": DESIGN[name][1],
                 "affects_observed": "both" if cohort_chg and label_chg else "cohort" if cohort_chg else "labels" if label_chg else "neither",
                 "cohort_n": len(ids), "early_excluded_n": len(early),
                 "left_cohort_n": len(left), "left_cohort_aki_pos": int(base_AKI.loc[left].sum()),
                 "entered_cohort_n": len(entered), "entered_cohort_aki_pos": int(v.loc[entered].sum()),
                 "common_stays_n": len(common), "labels_changed_common": int((b != x).sum()),
                 "neg_to_pos": int((~b & x).sum()), "pos_to_neg": int((b & ~x).sum()),
                 "common_with_reference_n": len(set(common) & ref_ids)})
registry = pd.DataFrame(rows)
registry.to_csv(OUTPUT / "variant_registry.csv", index=False)
print(registry.to_string(index=False))

#### Prognostic validity (landmark at 48 h)

In [ ]:

adm = pd.read_csv(HOSP / "admissions.csv", usecols=["hadm_id", "deathtime"], parse_dates=["deathtime"])
outc = cohort[["stay_id", "hadm_id", "intime", "outtime"]].merge(adm, on="hadm_id", how="left")
#landmark population: alive at 48 h, so the outcome follows the label window
outc = outc[outc["deathtime"].isna() | (outc["deathtime"] >= outc["intime"] + pd.Timedelta(hours=48))].copy()
outc["died"] = outc["deathtime"].notna().astype(int)



In [ ]:

from sklearn.metrics import roc_auc_score
def prognostic_row(name, aki_labels):
    d = aki_labels[["stay_id", "AKI"]].merge(outc[["stay_id", "died"]], on="stay_id", how="inner")
    a, b_ = d[d.AKI], d[~d.AKI]
    a1, a0, b1, b0 = a.died.sum(), len(a) - a.died.sum(), b_.died.sum(), len(b_) - b_.died.sum()
    OR = (a1 * b0) / (a0 * b1)
    se = np.sqrt(1/a1 + 1/a0 + 1/b1 + 1/b0)
    return dict(Variant=name, n=len(d), mort_AKI=round(a.died.mean()*100, 2), mort_noAKI=round(b_.died.mean()*100, 2),
                OR=round(OR, 2), CI_low=round(np.exp(np.log(OR)-1.96*se), 2), CI_high=round(np.exp(np.log(OR)+1.96*se), 2),
                AUC=round(roc_auc_score(d.died, d.AKI.astype(int)), 3))
prog = pd.DataFrame([prognostic_row(n, r[0]) for n, r in all_results.items()])
prog.to_csv(OUTPUT / "prognostic_validity_by_variant.csv", index=False)

In [ ]:
#by robustness group, using all_labels_matrix
grp = np.select([all_labels_matrix["fraction_positive"] == 0, all_labels_matrix["fraction_positive"] == 1],
                ["robust negative", "robust positive"], "fragile")
g = pd.DataFrame({"stay_id": all_labels_matrix.index, "group": grp}).merge(outc[["stay_id", "died"]], on="stay_id")
print(g.groupby("group")["died"].agg(n="size", mortality_pct=lambda x: round(x.mean()*100, 2)))
print(prog.to_string(index=False))

In [ ]:

ids = set(all_results["baseline"][1]["stay_id"])
ev = outputevents[outputevents["stay_id"].isin(ids)]
e_ = generate_urine_labels_elapsed_time(ev, weight, window_start=24, window_end=48, max_gap_hours=None)
k_ = generate_urine_labels_consecutive(ev, weight, window_start=24, window_end=48)
mc = e_.merge(k_, on="stay_id", suffixes=("_e", "_k"))
both = mc.dropna(subset=["urine_evaluable_et_e", "urine_evaluable_et_k"])
ae, bk = both.urine_evaluable_et_e.astype(bool), both.urine_evaluable_et_k.astype(bool)
print("stays compared:", len(mc), "| with an evaluability value under both rules:", len(both))
print("evaluable mean rule:", int(ae.sum()), "| consecutive rule:", int(bk.sum()), "| disagreements:", int((ae != bk).sum()))
print("elapsed positives:", int(mc.urine_labels_et_e.fillna(False).astype(bool).sum()),
      "| consecutive positives:", int(mc.urine_labels_et_k.fillna(False).astype(bool).sum()))
print("consecutive+ but elapsed-:", int((mc.urine_labels_et_k.fillna(False).astype(bool) & ~mc.urine_labels_et_e.fillna(False).astype(bool)).sum()))


In [ ]:
d = all_results["baseline"][0].merge(refs, on="stay_id")
d = d[~d["urine_evaluable"]].copy()
d["no_weight"] = ~d["stay_id"].isin(weight["stay_id"])
print("n compared:", len(d), "| comparator positive:", int(d["AKI_ref"].sum()),
      "| urine-positive:", int(d["AKI_ref_uo"].sum()), "| creatinine-positive:", int(d["AKI_ref_creat"].sum()))
print("comparator urine-positive but creatinine-negative:", int((d["AKI_ref_uo"] & ~d["AKI_ref_creat"]).sum()))
print(d.groupby("no_weight")[["AKI", "AKI_ref", "AKI_ref_uo"]].agg(["size", "sum"]))

In [ ]:
b = all_results["baseline"][0].set_index("stay_id")
for name in ["weight_full_stay", "weight_durations_ref"]:
    v = all_results[name][0].set_index("stay_id")
    common = sorted(set(v.index) & set(b.index))
    print(name, len(common))
    print(pd.crosstab(b.loc[common, "AKI"].astype(bool), v.loc[common, "AKI"].astype(bool)))

In [ ]:
common_ids = set(all_results["baseline"][0]["stay_id"]) & set(all_results["rolling_baseline"][0]["stay_id"])
for name in ["baseline", "rolling_baseline"]:
    lab = all_results[name][0]
    for tag, d in [("own cohort", lab), ("common stays", lab[lab["stay_id"].isin(common_ids)])]:
        c = d.merge(ref_label_creat[["stay_id", "AKI_ref_creat"]], on="stay_id")
        print(f"{name:17s} {tag:13s} n={len(c)}  FP={int((~c.AKI_ref_creat & c.aki_scr).sum())}  FN={int((c.AKI_ref_creat & ~c.aki_scr).sum())}")

In [ ]:
strict = labevents_with_hadm[labevents_with_hadm["itemid"] == 50912].sort_values(["stay_id", "charttime"]).copy()
strict["hours_from_icu"] = (strict["charttime"] - strict["intime"]).dt.total_seconds() / 3600
creatinine_v5 = creatinine
try:
      creatinine = strict
      abl = run_variant("abl_strict_extraction", VARIANTS["baseline"])
finally:
      creatinine = creatinine_v5
c = abl[0].merge(refs[["stay_id", "AKI_ref_creat"]], on="stay_id")
print("creatinine κ:", round(kappa_np(c["AKI_ref_creat"].to_numpy(bool), c["aki_scr"].to_numpy(bool)), 4), "n =", len(c))

In [ ]:
def fp_ids(name):
    c = all_results[name][0].merge(ref_label_creat[["stay_id", "AKI_ref_creat"]], on="stay_id")
    return set(c.loc[~c.AKI_ref_creat & c.aki_scr, "stay_id"])
fb, fc = fp_ids("baseline"), fp_ids("icu_censored")
print("v5 false positives:", len(fb), "| still false positive after censoring:", len(fb & fc), "| removed by censoring:", len(fb - fc))

In [ ]:
b = set(all_results["baseline"][1]["stay_id"]); v = set(all_results["early_excl_wd_weight"][1]["stay_id"])
extra = v - b
print(len(extra), "| early-excluded in v5:", len(extra & set(all_results["baseline"][2])))

In [ ]:
a = abl[0]; b = all_results["baseline"][0]
common = sorted(set(a["stay_id"]) & set(b["stay_id"]))
res = {}
for tag, lab in [("v5", b), ("strict extraction", a)]:
    c = lab[lab["stay_id"].isin(common)].merge(refs[["stay_id", "AKI_ref_creat"]], on="stay_id")
    res[tag] = kappa_np(c["AKI_ref_creat"].to_numpy(bool), c["aki_scr"].to_numpy(bool))
    print(tag, "n =", len(c), "creatinine kappa =", round(res[tag], 4))
print("extraction contribution (paired):", round(res["v5"] - res["strict extraction"], 4))

In [ ]:
extra = set(all_results["early_excl_wd_weight"][1]["stay_id"]) - set(all_results["baseline"][1]["stay_id"])
w_v5 = weight.set_index("stay_id")["weight_kg"]
w_wd = compute_weight_from_durations(weight_durations_raw, cohort).set_index("stay_id")["weight_kg"]
j = pd.DataFrame({"v5": w_v5.reindex(list(extra)), "wd": w_wd.reindex(list(extra))})
print("with v5 weight:", int(j["v5"].notna().sum()), "| with wd weight:", int(j["wd"].notna().sum()))
print("wd heavier than v5:", int((j["wd"] > j["v5"]).sum()), "| wd lighter:", int((j["wd"] < j["v5"]).sum()), "| no wd weight:", int(j["wd"].isna().sum()))

In [ ]:
import json, platform, hashlib, sklearn, datetime
def _sha(p): return hashlib.sha256(Path(p).read_bytes()).hexdigest()
log = {"date": str(datetime.date.today()), "seed": SEED, "n_boot": N_BOOT, "rerun_flag": RERUN,
       "bootstrap": "paired stay-level percentile bootstrap, 1000 resamples, on stays common to V5 and variant that have a reference label",
       "python": platform.python_version(), "pandas": pd.__version__, "numpy": np.__version__,
       "sklearn": sklearn.__version__, "duckdb": duckdb.__version__,
       "kdigo_pipeline_sha256": _sha(PROJECT_DIR / "kdigo_pipeline.py"),
       "labels_v5_sha256": _sha(OUTPUT / "labels_v5.parquet"),
       "derived_files": {f: _sha(DATA / "derived" / f) for f in ["kdigo_stages.csv", "kdigo_creatinine.csv", "kdigo_uo.csv", "weight_durations.csv", "crrt.csv"]},
          "duckdb_modified": (datetime.datetime.fromtimestamp(Path("mimic-iv.duckdb").stat().st_mtime).isoformat()
                       if Path("mimic-iv.duckdb").exists() else None),
       "n_results": len(all_results)}
with open(OUTPUT / "Sensitivity_Analysis" / "run_log_sensitivity.json", "w") as f:
    json.dump(log, f, indent=2)